```17/10/2026```

# Poincaré Map tests

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from emerald.classical.dynamics import ClassicalSystem, Dynamics, StaticDynamics
from emerald.fields import SinusoidField
from emerald.potentials import Morse

In [ ]:
mor = Morse(D=1, beta=1)

In [ ]:
E0 = -0.5
J0 = mor.action(E0)
w0 = mor.angular_frequency(E0)

# -1/2, √2-1, 1
E0, J0, w0

In [ ]:
r0 = 1
p0 = mor.momentum(E0, r0)

In [ ]:
fld = SinusoidField(0.03, 1, 0)

dyn = Dynamics(mor, fld)

In [ ]:
systs = []

for _E in np.linspace(-0.1, -0.9, 21):
    systs.append(ClassicalSystem.from_angle(dyn, _E, N=30))

syst = ClassicalSystem.merge_systems(dyn, systs)

In [ ]:
plt.scatter(*syst.states0.T, s=10, marker='.')

In [ ]:
pmap = syst.poincare_map(100)

In [ ]:
%matplotlib qt5

In [ ]:
plt.plot(*mor.phase_space(E0, 500), ls='dashed', c='grey')
plt.plot(*mor.phase_space(-0.3, 500), ls='dashed', c='grey')
plt.scatter(pmap.position, pmap.momentum, s=1, marker='.')
# plt.scatter(pmap.angle, pmap.energy, s=5, marker='.')

In [ ]:
mor.angular_frequency(-0.5)

In [ ]:
r0 = 1.92639
p0 = 0.12428
# p0 = mor.momentum(-0.38, r0)

traj = dyn.propagate(np.array([r0, p0]), (0, 10*fld.period), np.linspace(0, 10*fld.period, 1000))

In [ ]:
mor.angular_frequency(-0.3)

In [ ]:
# plt.plot(traj.time, traj.position)
plt.plot(traj.position, traj.momentum)

In [ ]:
pmap.angle.shape

In [ ]:
pmap.action.shape

In [ ]:
mor.period(E0)

In [ ]:
static_dyn = StaticDynamics(mor)

In [ ]:
fld = SinusoidField(0.25, mor.angular_frequency(-0.6)/np.pi, 0)

In [ ]:
dyn = Dynamics(mor, fld)

In [ ]:
b = 0.04 ; a = 0.09

thetas = np.linspace(0, 2*np.pi, 1000)

rs = a*np.cos(thetas)+1
ps = b*np.sin(thetas)

# rs = np.append(rs, rs)
# ps = np.append(ps, -ps)

plt.scatter(rs, ps)

In [ ]:
states0 = np.column_stack((rs, ps))

syst_stability = ClassicalSystem(dyn, states0)

In [ ]:
traj = syst_stability.propagate(np.linspace(0, 50, 500))

In [ ]:
plt.plot(*mor.phase_space(-0.6, 500), ls='dashed', c='grey')
plt.plot(*mor.phase_space(-0.5593914567730168, 500), ls='dashed', c='grey')
plt.plot(*mor.phase_space(-0.6430224751679855, 500), ls='dashed', c='grey')
i = 499
plt.plot(traj.position[i, :], traj.momentum[i, :], label=f"$t={traj.time[i]:.2f}$")
plt.plot(traj.position[0, :], traj.momentum[0, :], c='black', label="Initial Ensemble")
plt.legend()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Figure and axes
fig, ax = plt.subplots(figsize=(9, 6), layout="constrained")

# Reference phase-space curves
for E in [-0.6, -0.5593914567730168, -0.6430224751679855]:
    ax.plot(
        *mor.phase_space(E, 500),
        ls="--", c="grey", lw=1.0, alpha=0.7
    )

# Initial ensemble (static)
ax.plot(
    traj.position[0, :],
    traj.momentum[0, :],
    c="black", lw=1.0, alpha=0.5,
    label="Initial Ensemble"
)

# Animated ensemble
line, = ax.plot(
    [], [],
    lw=1.4, c="tab:blue",
    label=""
)

# Formatting
ax.set_xlabel(r"$r$")
ax.set_ylabel(r"$p$")
ax.set_title("Phase-space evolution")
ax.grid(True, which="major", ls=":", alpha=0.5)
ax.minorticks_on()
ax.grid(True, which="minor", ls=":", alpha=0.2)
ax.axhline(0, color="black", lw=0.6, alpha=0.4)
ax.axvline(0, color="black", lw=0.6, alpha=0.4)
line, = ax.plot([], [], lw=1.4, c="tab:blue", label=r"$t=0.00$")
legend = ax.legend(loc="best", framealpha=0.9)

# Animation callbacks
def init():
    line.set_data([], [])
    legend.get_texts()[1].set_text("")
    return line, *legend.get_texts()

def update(i):
    line.set_data(
        traj.position[i, :],
        traj.momentum[i, :]
    )

    line.set_label(rf"$t={traj.time[i]:.2f}$")
    legend.get_texts()[1].set_text(line.get_label())

    return line, *legend.get_texts()

ani = FuncAnimation(
    fig,
    update,
    frames=range(len(traj.time)),
    init_func=init,
    interval=30,       # milliseconds between frames
    blit=False,
    repeat=False
)

plt.show()

In [ ]:
ani.save("phase_space_field.mp4", writer="ffmpeg", fps=30)

In [ ]:
r, p = states0[int(len(states0)/2)]
p**2/2 + mor(r)

In [ ]:
def polygon_area(points):
    x = points[:, 0]
    y = points[:, 1]
    
    # Vectorized Shoelace calculation
    return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))


In [ ]:
polygon_area(states0)

In [ ]:

areas = [polygon_area(state) for state in traj.states]

plt.plot(areas)
plt.ylim(0, 0.012)

In [ ]:
from emerald.classical.dynamics import (
    _DEFAULT_REGULARIZED_INTEGRATOR,
    RegularizedDynamics,
)
from emerald.potentials.coulomb import Coulomb

cou = Coulomb()
dyn_cou = RegularizedDynamics(cou, fld)

syst_cou = ClassicalSystem.from_angle(dyn_cou, -0.5, 80, position_signs=1)

In [ ]:
%matplotlib qt5

In [ ]:
plt.scatter(*syst_cou.states0.T)

In [ ]:
cou_pmap = syst_cou.poincare_map(200, integrator=_DEFAULT_REGULARIZED_INTEGRATOR)

In [ ]:
plt.scatter(cou_pmap.angle, cou_pmap.energy, s=5, marker='.')